# ปฏิบัติการสัปดาห์ที่ 1: Introduction to Health Informatics & Data Pre-processing
### วิชา อจวพ 302 ปฏิบัติการสารสนเทศด้านสุขภาพ (AMMS 302 Workshop in Health Informatics)

ในปฏิบัติการคาบแรกนี้ นักศึกษาจะได้เรียนรู้วิธีการใช้ภาษา Python และไลบรารี **pandas** ในการจัดการและตระเตรียมข้อมูลผู้ป่วยจำลอง (Data Pre-processing) ก่อนนำไปใช้ในการวิเคราะห์ข้อมูลขนาดใหญ่และการแพทย์แม่นยำต่อไป

## 🛠️ Setup สำหรับ Windows ด้วย scoop + uv (Self-study, รันออฟไลน์ได้)

> **ทำตามได้บน Windows 10/11 โดยไม่ต้องพึ่งเซิร์ฟเวอร์มหาวิทยาลัย** — ใช้ PowerShell (ไม่ใช่ CMD)

**1. ติดตั้ง scoop (ครั้งเดียว):**
```powershell
Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser
Invoke-RestMethod -Uri https://get.scoop.sh | Invoke-Expression
scoop --version   # ทดสอบว่าติดตั้งสำเร็จ
```
คู่มือทางการ: [scoop.sh](https://scoop.sh) | [Scoop docs (GitHub)](https://github.com/ScoopInstaller/Scoop)

**2. ติดตั้ง git + uv ผ่าน scoop:**
```powershell
scoop install git uv
uv --version
```
เอกสารทางการ: [uv installation](https://docs.astral.sh/uv/getting-started/installation/) | [uv — managing projects](https://docs.astral.sh/uv/guides/projects/)

**3. สร้างโปรเจกต์และติดตั้งไลบรารี (ในโฟลเดอร์ health-informatics ของคุณ):**
```powershell
D:
mkdir health-informatics; cd health-informatics
uv init
uv add jupyterlab pandas faker tietai-synthea
# หรือถ้ามีโปรเจกต์อยู่แล้ว: uv add tietai-synthea
```

**4. เปิด JupyterLab แบบ reproducible:**
```powershell
uv run jupyter lab
# เปิดเบราว์เซอร์ที่ http://localhost:8888 แล้วเปิดไฟล์ .ipynb นี้
```

**หมายเหตุ PySynthea (แทน Synthea Java เดิม):**
- PyPI: [tietai-synthea](https://pypi.org/project/tietai-synthea/) — import ชื่อ `synthea`, คำสั่ง CLI `synthea`
- GitHub: [TIET-AI/tietai-synthea](https://github.com/TIET-AI/tietai-synthea) (Quick Start, API, disease modules)
- Paper: [PySynthea (PDF)](https://tiet.ai/pdf/pysynthea-paper.pdf) | Blog: [tiet.ai/blog/py-synthea](https://tiet.ai/blog/py-synthea/)
- ไม่ต้องติดตั้ง Java/JVM — รันด้วย `uv run synthea -p 10` หรือ Python API `from synthea import Generator, GeneratorOptions`
- ข้อมูลสังเคราะห์จะส่งออกเป็น FHIR R4 JSON Bundle — สเปกทางการ: [HL7 FHIR R4](https://www.hl7.org/fhir/) | [FHIR Patient](https://www.hl7.org/fhir/patient.html) | [FHIR Bundle](https://www.hl7.org/fhir/bundle.html)

**อ้างอิง pandas / Jupyter ที่ใช้ในแล็บนี้:**
- [pandas docs](https://pandas.pydata.org/docs/) | [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) | [pandas I/O](https://pandas.pydata.org/pandas-docs/stable/reference/io.html) | [Indexing & selecting](https://pandas.pydata.org/docs/user_guide/indexing.html)
- [JupyterLab docs](https://jupyterlab.readthedocs.io/en/latest/)

> ทุกเซลล์ในโน้ตบุ๊กนี้รันด้วย `uv run` จึงล็อกเวอร์ชันผ่าน `uv.lock` — ส่งโปรเจกต์ให้อาจารย์แล้วรันซ้ำได้ผลเดิม (reproducible)


## ขั้นตอนที่ 1: การนำเข้าไลบรารีหลัก
เราจะเริ่มด้วยการนำเข้าไลบรารี `pandas` และ `numpy` ซึ่งเป็นเครื่องมือมาตรฐานสำหรับวิทยาการข้อมูล

In [ ]:
import pandas as pd
import numpy as np
print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)

## ขั้นตอนที่ 2: การเปิดอ่านและตรวจสอบข้อมูลผู้ป่วยดิบ
เราจะลองเปิดอ่านไฟล์ `patients_data.csv` ที่ระบบเตรียมไว้ให้ในโฟลเดอร์ปฏิบัติการ

In [ ]:
# โหลดข้อมูลจากไฟล์ CSV
df = pd.read_csv('patients_data.csv')

# แสดงข้อมูล 5 แถวแรก
print(f"ขนาดตารางข้อมูล: {df.shape[0]} แถว, {df.shape[1]} คอลัมน์")
df.head()

## ขั้นตอนที่ 3: การจัดการค่าสูญหาย (Handling Missing Values)
ข้อมูลทางการแพทย์จริงมักมีช่องว่าง (Null/NaN) เนื่องจากไม่ได้ตรวจวัดในคนไข้ทุกคน เราจะตรวจสอบและจัดการค่าว่างเหล่านี้

In [ ]:
# 1. ตรวจสอบจำนวนค่าว่างในแต่ละคอลัมน์
print("จำนวนค่าว่างในแต่ละตัวแปร:")
print(df.isnull().sum())

# 2. ตัวอย่างการคัดแถวที่ไม่มีข้อมูลแล็บสำคัญออก (เช่น ค่า HbA1c_level ว่าง)
df_clean_hba1c = df.dropna(subset=['HbA1c_level'])
print(f"จำนวนแถวหลังจากลบค่า HbA1c ว่าง: {len(df_clean_hba1c)} แถว")

# 3. ตัวอย่างการแทนค่าว่างด้วยค่าเฉลี่ยสถิติของกลุ่ม (Mean Imputation)
mean_bp = df['systolic_bp'].mean()
df['systolic_bp'] = df['systolic_bp'].fillna(mean_bp)
print(f"แทนที่ค่า systolic_bp ว่างด้วยค่าเฉลี่ย: {mean_bp:.2f} mmHg สำเร็จ")

## ขั้นตอนที่ 4: การคัดกรองข้อบกพร่องทางสรีรวิทยา (Physiological Validity Checks)
คนไข้ไม่สามารถมีอายุติดลบได้ และในมาตรฐาน HIPAA มักจำกัดการระบุตัวตนผู้ป่วยสูงอายุมาก (อายุ > 89) เราจะทำการคัดกรองข้อมูลที่ขัดต่อสรีรวิทยาเหล่านี้ออก

In [ ]:
# 1. กรองข้อมูลเฉพาะผู้ป่วยที่มีอายุตั้งแต่ 0 ถึง 89 ปี
df_valid_age = df[(df['Age'] >= 0) & (df['Age'] <= 89)]
print(f"จำนวนผู้ป่วยที่อยู่ในเกณฑ์อายุสมบูรณ์: {len(df_valid_age)} คน (จากทั้งหมด {len(df)} คน)")

# 2. ตรวจสอบข้อมูลความดันโลหิตบีบตัว (systolic_bp) ที่เป็นไปได้ทางสรีรวิทยา (เช่น ไม่เกิน 220 mmHg)
df_valid = df_valid_age[df_valid_age['systolic_bp'] <= 220]
print(f"จำนวนผู้ป่วยหลังกรองค่าความดันโลหิตบกพร่อง: {len(df_valid)} คน")

## ขั้นตอนที่ 5: การวิเคราะห์สืบค้นกลุ่มผู้ป่วย (Cohort Filtering)
เราต้องการกรองหาผู้ป่วยเฉพาะกลุ่ม เช่น ผู้ป่วยสูงอายุที่มีความเสี่ยงสะสมรุนแรง (อายุ > 60 ปี และมีระดับน้ำตาลสะสม HbA1c_level > 7.0%)

In [ ]:
# กรองกลุ่มเป้าหมาย (Cohort Filtering)
high_risk_cohort = df_valid[(df_valid['Age'] > 60) & (df_valid['HbA1c_level'] > 7.0)]
print(f"พบผู้ป่วยเบาหวานสูงอายุกลุ่มเสี่ยงสูงจำนวน: {len(high_risk_cohort)} ราย")
high_risk_cohort.head()

## ขั้นตอนที่ 6: การปรับปรุงรูปแบบชนิดข้อมูล (Data Type & DateTime Conversion)
วันที่ที่โหลดเข้ามาเป็นเพียงข้อความตัวหนังสือ (object) เราต้องแปลงให้เป็นชนิดข้อมูลวันเวลา (Datetime) เพื่อคำนวณระยะเวลารักษาได้ถูกต้อง

In [ ]:
# ตรวจสอบชนิดข้อมูลเดิม
print("ชนิดข้อมูลเดิม:")
print(df_valid[['Age', 'admission_date']].dtypes)

# แปลงวันที่ด้วย pd.to_datetime()
df_valid = df_valid.copy()
df_valid['admission_date'] = pd.to_datetime(df_valid['admission_date'])
print("\nชนิดข้อมูลหลังการแปลง:")
print(df_valid[['Age', 'admission_date']].dtypes)

## ขั้นตอนที่ 7: การส่งออกไฟล์ผลลัพธ์ข้อมูลสะอาด (Saving Intermediate Work)
การเก็บบันทึกข้อมูลที่ทำความสะอาดแล้วเพื่อความปลอดภัยเชิงระบบและการคำนวณที่ง่ายขึ้น

In [ ]:
# ส่งออกเป็นตาราง CSV สะอาด
df_valid.to_csv('cleaned_patients_data.csv', index=False)
print("บันทึกไฟล์ cleaned_patients_data.csv เรียบร้อยแล้ว!")

## ขั้นตอนที่ 8: ปฏิบัติการสืบค้นข้อมูลกึ่งโครงสร้างสากล (JSON/FHIR Resource)
ทดลองเปิดและสืบค้นคีย์ของข้อมูลกึ่งโครงสร้างแบบ JSON ตามมาตรฐาน HL7 FHIR ในสไลด์การบรรยายหน้า 15-16

In [ ]:
import json

# 1. เปิดอ่านข้อมูลผู้ป่วยกึ่งโครงสร้าง
with open('patient_fhir_demo.json', 'r', encoding='utf-8') as file:
    record = json.load(file)

# 2. ดึงข้อมูลส่วนตัวระดับบุคคลแบบระบุคีย์ตรงๆ (สไลด์ที่ 15)
patient_id = record['id']
resource_type = record['resourceType']
print(f"รหัสทรัพยากร: {patient_id} | ประเภท: {resource_type}")

# 3. ดึงข้อมูลผู้ป่วยรายแรกใน entry
first_entry = record['entry'][0]['resource']
given_name = first_entry['name'][0]['given'][0]
family_name = first_entry['name'][0]['family']
birth_date = first_entry['birthDate']
print(f"คนไข้ FHIR รายแรก: {given_name} {family_name} | วันเกิด: {birth_date}")

# 4. คลี่โครงสร้าง JSON ซับซ้อนให้กลายเป็นตาราง Pandas (สไลด์ที่ 16)
df_fhir = pd.json_normalize(record, record_path=['entry'])
df_fhir.head()

### 🧬 เสริม: สังเคราะห์ผู้ป่วยด้วย PySynthea (Python-native, แทน Synthea Java) — รันออฟไลน์บน Windows

เซลล์นี้สาธิตการสร้างข้อมูลผู้ป่วยสังเคราะห์แบบ FHIR R4 ด้วย **PySynthea** แล้วโหลดเข้า pandas ต่อทันที (ไม่ต้องใช้ Java)

> ถ้ายังไม่ได้ `uv add tietai-synthea` ให้รันใน PowerShell ก่อน: `uv add tietai-synthea` แล้วรีสตาร์ท kernel ด้วย `uv run jupyter lab`

เอกสารทางการ: [PySynthea GitHub — Quick Start](https://github.com/TIET-AI/tietai-synthea#quick-start) | [PyPI](https://pypi.org/project/tietai-synthea/) | [Paper PDF](https://tiet.ai/pdf/pysynthea-paper.pdf) | FHIR สเปก: [hl7.org/fhir](https://www.hl7.org/fhir/)


In [ ]:
# --- PySynthea demo (Python-native, ไม่ต้องติดตั้ง Java) ---
# ต้องการ: uv add tietai-synthea  (แล้วรันโน้ตบุ๊กด้วย uv run jupyter lab)
try:
    from synthea import Generator, GeneratorOptions
    import tempfile, pathlib, json, pandas as pd

    # 1. ตั้งค่าการสังเคราะห์ (แก้ไข state/city/seed ได้ตามต้องการ)
    options = GeneratorOptions()
    options.population_size = 5          # สร้าง 5 คนพอสำหรับเดโม (เพิ่มได้ 10-100)
    options.state = "California"
    options.city = "San Francisco"
    options.seed = 42

    # 2. รัน generator (ใช้เวลาหลักวินาทีสำหรับ 5 คน)
    generator = Generator(options)
    generator.run()
    print("Generated stats:", generator.stats)

    # 3. ส่งออก FHIR R4 JSON ไปโฟลเดอร์ชั่วคราว แล้วโหลดด้วย pandas
    #    (PySynthea มัดรวม disease modules + demographics ไว้ในแพ็กเกจแล้ว — ไม่ต้อง clone repo)
    tmpdir = pathlib.Path(tempfile.mkdtemp())
    # ใช้ exporter ของ PySynthea; ถ้า API เปลี่ยน ให้ดู https://github.com/TIET-AI/tietai-synthea#python-api-usage
    try:
        from synthea.export.fhir import FHIRExporter
        exporter = FHIRExporter(generator.config, tmpdir)
        # ตัวอย่าง: ส่งออกผู้ป่วยคนแรก (API อาจต่างกันเล็กน้อยตามเวอร์ชัน — อ่าน docstring)
        print(f"FHIR export dir: {tmpdir} — ดูไฟล์ *.json แล้วฝึก pd.json_normalize ต่อได้")
    except Exception as e:
        print("Exporter demo note:", e)
        print("ทางเลือก: รัน CLI แทน — uv run synthea -p 10 แล้วดูโฟลเดอร์ output/fhir/")

    # 4. ตัวอย่าง: โหลด Bundle ที่ได้มาแล้วคลี่เป็นตาราง (pattern เดียวกับ patient_fhir_demo.json ในแล็บ)
    #    ถ้ายังไม่มีไฟล์จาก generator ให้ข้ามไปใช้ patient_fhir_demo.json ที่มีอยู่แล้วในโฟลเดอร์นี้
    demo_fhir_path = pathlib.Path("patient_fhir_demo.json")
    if demo_fhir_path.exists():
        with open(demo_fhir_path, encoding="utf-8") as f:
            bundle = json.load(f)
        df_demo = pd.json_normalize(bundle, record_path=["entry"])
        print("Demo bundle -> DataFrame shape:", df_demo.shape)
        display(df_demo.head())

except ModuleNotFoundError:
    print("ยังไม่ได้ติดตั้ง tietai-synthea — รันใน PowerShell: uv add tietai-synthea แล้วรีสตาร์ท kernel")
except Exception as e:
    print("PySynthea demo error (ดูเอกสาร https://github.com/TIET-AI/tietai-synthea):", e)


## การบ้านสัปดาห์ที่ 1 (Homework 1 - 20 คะแนนสะสม)
**โจทย์ปฏิบัติการส่วนบุคคล (ประเมินผลสัมฤทธิ์ CLO2):**
1. ให้นักศึกษาเปิดตารางข้อมูลผู้ป่วยดิบ **`raw_patients.csv`** ซึ่งเป็นข้อมูลดิบที่มีจุดบกพร่องอยู่
2. ตรวจหาจำนวนค่าว่าง (Null) ทั้งหมดในไฟล์และแสดงสรุปผลทางหน้าจอ
3. ทำความสะอาดค่าสูญหายในคอลัมน์ระดับน้ำตาลสะสมสะสม `HbA1c_level` ด้วยการแทนที่ด้วย **ค่าเฉลี่ยสถิติ (Mean)**
4. เขียนเงื่อนไขกรองกำจัดผู้ป่วยที่มีค่าอายุติดลบ หรืออายุเกินสรีรวิทยาที่เป็นไปได้จริง (มากกว่า 89 ปี)
5. กรองเลือกเฉพาะผู้ป่วยเพศหญิง (**`Gender == 'หญิง'`**) ที่มีความเสี่ยงเป็นเบาหวานสะสม (**`HbA1c_level > 6.5`**)
6. เก็บบันทึกและส่งออกผลลัพธ์เป็นไฟล์ CSV ชื่อ **`student_id_homework1.csv`** และนำส่งไฟล์ `.ipynb` บนระบบมหาวิทยาลัยก่อนเริ่มคาบเรียนถัดไป